# T038 · 蛋白质–配体相互作用预测

本教程属于 TeachOpenCADD。该平台提供领域技能教程和可用于起步的研究流程模板。

原作者

- Roman Joeres，2022，萨尔兰大学，[UdS 与 HIPS 药物生物信息学教席](https://www.helmholtz-hips.de/de/forschung/teams/team/wirkstoffbioinformatik/)，[NextAID](https://nextaid.cs.uni-saarland.de/) 项目

*中文导读依据 TeachOpenCADD 的 [上游 Notebook](https://github.com/volkamerlab/teachopencadd/blob/834b9ff59af69e8ba8d42aae23e92c15eb734ef0/teachopencadd/talktorials/T038_protein_ligand_interaction_prediction/talktorial.ipynb) 编译，沿用原课的 [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) 许可。*

## 本课目标

学习使用图神经网络（GNN）预测蛋白质–配体相互作用。GNN 可以把蛋白质结构和小分子配体这类带连接关系的数据编码为向量。本课将建立图表示，并训练一个模型，预测给定蛋白质与配体是否相互作用。

### 理论内容

- 蛋白质–配体相互作用预测的意义
- 模型流程
- 生物学背景：蛋白质的图表示
- 技术背景
  - 图同构网络
  - 二元交叉熵损失

### 实践内容

- 构造图表示
  - 将配体转换为分子图
  - 将蛋白质转换为残基图
- 管理数据
  - 样本
  - 数据集
  - 数据模块
- 实现网络
  - GNN 编码器
  - 完整模型
- 训练流程

### 参考资料

- **理论背景**
  - 图神经网络
    - Kipf 与 Welling，《基于图卷积网络的半监督分类》，[<i>arXiv</i>](https://arxiv.org/abs/1609.02907)
    - Bronstein 等，《几何深度学习：从欧氏数据拓展到非欧氏数据》，[<i>IEEE Signal Processing Magazine</i>（2017），18–42](https://doi.org/10.1109/MSP.2017.2693418)
  - 基于深度学习和 GNN 的药物–靶点结合预测
    - Öztürk 等，《DeepDTA：药物–靶点结合亲和力的深度学习预测》，[<i>Bioinformatics</i>（2018），<b>34</b>，i821–i829](https://doi.org/10.1093/bioinformatics/bty593)
    - Nguyen 等，《GraphDTA：使用图神经网络预测药物–靶点结合亲和力》，[<i>Bioinformatics</i>（2021），<b>37</b>，1140–1147](https://doi.org/10.1093/bioinformatics/btaa921)
  - 图同构网络
    - Xu 等，《图神经网络有多强？》，[<i>arXiv</i>（2018）](https://arxiv.org/abs/1810.00826)
- **实践资料**
  - [PyTorch](https://pytorch.org/)
  - [PyTorch Geometric](https://pytorch-geometric.readthedocs.io/en/latest/)
  - [RDKit](http://rdkit.org/) 与 Greg Landrum 的[分子指纹讲义](https://www.rdkit.org/UGM/2012/Landrum_RDKit_UGM.Fingerprints.Final.pptx.pdf)

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install teachopencadd --no-deps -q
    !teachopencadd -d 38
    %pip uninstall teachopencadd -y -q
    %pip install -qr requirements.txt

## 理论

本课将此前的分子图表示、图神经网络和监督学习组合起来，用于预测蛋白质和配体是否会相互作用。下文在相关概念旁注明对应课号。

### 为什么预测蛋白质–配体相互作用？

相互作用分析在药物发现等研究中广泛应用（见 **T016**）。面对给定的靶蛋白，实验筛选可以进行逐个测定并按结合强弱排序。结合亲和力描述两个分子的结合强度。虚拟筛选用计算方法预先筛查大量配体，可以把实验资源优先用于更值得复核的候选。

本课将问题定义为定性二分类，预测一对蛋白质–配体是否结合。结合亲和力的定量回归可以作为另一个任务。

### 模型结构

训练数据由蛋白质、配体及其配对标签表组成。我们沿用 **T022** 的监督学习框架，把每对样本分为结合与非结合两类。

模型包含两个独立的 GNN 编码器，分别把蛋白质图和配体图编码为固定长度的向量（见 **T035**）。将两个向量拼接，再输入一个多层感知机（MLP，见 **T022**），得到结合预测分数。

![由两个图编码器和一个 MLP 构成的模型](./images/basic_structure_nn.png)

**图 1.** 本课模型结构示意图。例图使用 PDB 条目 [4O75](https://www.rcsb.org/structure/4O75) 的结构，PDB 使用方法见 **T008**。

### 生物学背景：将蛋白质表示为图

本节重点介绍蛋白质的图表示。通过 SMILES 建立配体分子图的原理见 **T033**。

蛋白质可以用氨基酸序列或三维结构来描述。序列提供残基顺序，PDB 结构同时提供原子坐标（见 **T008**），因此适合提取跨序列远距离的空间邻接关系。我们用每个残基的主链 α 碳（$C_\alpha$）代表其位置。如果两个残基的 $C_\alpha$ 足够接近，就在它们对应的两个节点之间连边。这条边表示按距离定义的邻接关系。

![用残基的 α 碳构建蛋白质接触图](./images/prot_graph_creation.png)

**图 2.** 残基接触图构造示意。图中只将 $C_\alpha$ 距离在 7 Å 阈值内的半胱氨酸标为相邻。半胱氨酸侧链的硫原子可以在合适的几何与化学条件下形成二硫键，帮助稳定折叠结构。距离阈值本身只定义图中的空间邻接关系。

![三种示例氨基酸的 α 碳原子](./images/calphas.jpg)

**图 3.** 三种氨基酸碳原子的示意图。本课用的是每个残基主链的 $C_\alpha$。 [图源](https://chemistry.stackexchange.com/questions/134409/what-exactly-makes-a-carbon-atom-%CE%B1-in-a-protein-residue)

### 数据划分

将训练、验证和测试分别放在独立子集，有助于评价泛化和检查过拟合。可用随机划分、按标签分层，或按样本的相似性划分。例如，可以用分子结构相似性（**T005**）和蛋白质序列相似性（**T024、T028**）来决定哪些配对应该放在同一子集。关于数据泄漏的讨论见 [Bernett 等，2024](https://doi.org/10.1038/s41592-024-02362-y) 和 [Joeres 等，2025](https://doi.org/10.1038/s41467-025-58606-8)。

**T007** 用 $k$ 折交叉验证划分数据。本课为演示流程，随机保留约 70% 的配对用于训练、20% 用于验证、10% 用于测试。有两类主要的重叠风险。

- 同一个蛋白质出现在训练和测试中，模型可能记住该靶点的特征。
- 同一个配体出现在训练和测试中，模型可能记住该配体的特征。

如果研究问题是迁移到新蛋白质或新化合物，测试集应按目标应用隔离对应实体，必要时还应控制跨子集的序列、结构或骨架相似性。

### 技术背景

下文说明 GNN 的结构和节点特征。激酶与配体编码器使用相同类型的网络层，但有各自的参数和不同的输入特征维度。

#### 图同构网络（Graph Isomorphism Network，GIN）

[PyTorch Geometric 卷积层列表](https://pytorch-geometric.readthedocs.io/en/latest/modules/nn.html#convolutional-layers)列出了多种 GNN 架构。本课以 `GINConv` 为骨干。一个节点的更新可以写成

$$\mathbf{x}^{\prime}_i = h_{\mathbf{\Theta}} \left( (1 + \epsilon) \cdot \mathbf{x}_i + \sum_{j \in \mathcal{N}(i)} \mathbf{x}_j \right).$$

其中 $\mathcal{N}(i)$ 是节点 $i$ 的邻居集合，$\epsilon$ 是本例中的常数超参数，$h_{\mathbf{\Theta}}$ 是神经网络（见 **T022**）。网络将邻居向量相加，与加权后的自身向量合并，再应用 $h_{\mathbf{\Theta}}$ 提取新特征。

这一版本的 `GINConv` 使用图的边索引来确定邻居，同时可以只提供节点特征，无需额外的边属性。每个节点用类别特征表示，例如氨基酸类型或原子元素，独热编码参见 **T021**。

最后使用均值池化，把最后一层所有节点的嵌入求平均，得到整张图的一个向量。

#### 二元交叉熵损失（BCE）

**T022** 介绍的 MSE 和 MAE 常用于回归。本课用[二元交叉熵损失](https://pytorch.org/docs/stable/generated/torch.nn.BCELoss.html)训练分类模型。

$$-\left[ y\cdot\log(x)+(1-y)\cdot\log (1-x)\right],$$

其中 $x$ 是模型对样本给出的结合概率，$y$ 是标签。若 $y=1$，损失为 $-\log(x)$，最小化损失会推动预测接近 1。若 $y=0$，损失为 $-\log(1-x)$，预测应接近 0。前面的负号把最大化对数似然改写成最小化损失。

实际编码中，$y=1$ 对应数据中的结合配对，$y=0$ 对应非结合配对。

## 实践

下面先导入依赖、设置路径，再把 KiBA 相互作用数据整理为图模型可以读取的格式。首次预处理可能访问在线数据库，应保留课目录中的 `utils` 和 `data`。

In [2]:
import math
import random
import os
from pathlib import Path

from rdkit import Chem
import torch
import torch.nn.functional as F
from torch import nn
from torch.optim import Adam
from torch_geometric.nn import global_mean_pool, GINConv
from torch_geometric.data import Data, InMemoryDataset
from torch_geometric.loader import DataLoader
import matplotlib.pyplot as plt

# reproducibility
torch.manual_seed(0)

# Use cuda if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using", device)

from utils import kiba_preprocessing

/home/rjo21/miniconda3/envs/trash/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using cuda


/home/rjo21/miniconda3/envs/trash/lib/python3.13/site-packages/chembl_webresource_client/__init__.py:4: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  __version__ = __import__('pkg_resources').get_distribution('chembl_webresource_client').version


In [3]:
HERE = Path(".")
DATA = HERE / "data"
IMGS = HERE / "images"

# This method calls a data-preprocessing pipeline that is very technical and not of bigger interest for this talktorial.
# The method basically converts the KiBA dataset from an excel table to a dataset of structures in the format that we need.
kiba_preprocessing(DATA / "KIBA.csv", DATA / "resources")

KiBA originally contains 52498 ligands and 468 proteins.
KiBA after dropping sparse rows contains 79 ligands and 468 proteins.
KiBA finally contains 79 ligands and 373 proteins.
Preprocessing ligands
After ligand availability analysis KiBA contains 76 ligands and 373 proteins.
Preprocessing ligands finished
Preprocessing proteins
After protein availability analysis KiBA contains 76 ligands and 275 proteins.
Preprocessing proteins finished
Preprocessing interactions
Finally, KiBA comprises 20475 interactions.
Preprocessing interactions finished


### 构建图表示

#### 将配体转换为分子图

用 RDKit 将 SMILES 解析为分子。在含 $N$ 个原子的配体中，节点特征是 $N\times F$ 矩阵，$F$ 为每个原子的特征维度。这里用独热向量编码元素，最后一位归纳未单列的元素。

PyTorch Geometric 把边按源和目标节点索引存为形状 $2\times M$ 的张量，即有 $M$ 条有向边。每条化学键在代码中加入两个方向。

In [4]:
# For every atom type we consider, map the symbol to a numerical value for one-hot encoding.
atoms_to_num = dict(
    (atom, i) for i, atom in enumerate(["C", "N", "O", "F", "P", "S", "Cl", "Br", "I"])
)


def atom_to_onehot(atom):
    """
    Return the one-hot encoding for an atom given its index in the atoms_to_num dict.

    Parameters
    ----------
    atom: str
        Atomic symbol of the atom to represent

    Returns
    -------
    torch.Tensor
        A one-hot tensor encoding the atoms features.
    """
    # initialize a 0-vector ...
    one_hot = torch.zeros(len(atoms_to_num) + 1, dtype=torch.float)
    # ... and set the according field to one, ...
    if atom in atoms_to_num:
        one_hot[atoms_to_num[atom]] = 1.0
    # ... the last field is used to represent atom types that do not have their own field in the one-hot vector
    else:
        one_hot[len(atoms_to_num)] = 1.0
    return one_hot


def smiles_to_graph(smiles):
    """
    Convert a molecule given as SDF file into a graph.

    Arguments
    ---------
    smiles: str
        Path to the file storing the structural information of the ligand

    Returns
    -------
    Tuple[torch.Tensor, torch.Tensor]
        A pair of node features and edges in the PyTorch Geometric format
    """
    # read in the molecule from an SDF file
    mol = Chem.MolFromSmiles(smiles)
    atoms, bonds = [], []
    # check if the molecule is valid
    if mol is None:
        print(smiles)
        return None, None

    # iterate over all atom, compute the feature vector and store them in a torch.Tensor object
    for atom in mol.GetAtoms():
        atoms.append(atom_to_onehot(atom.GetSymbol()))
    atoms = torch.stack(atoms)

    # iterate over all bonds in the molecule and store them in the PyTorch Geometric specific format in a torch.Tensor,
    for bond in mol.GetBonds():
        bonds.append((bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()))
        bonds.append((bond.GetEndAtomIdx(), bond.GetBeginAtomIdx()))
    bonds = torch.tensor(bonds, dtype=torch.long).T

    return atoms, bonds

#### 将蛋白质转换为残基图

同样返回节点特征与边索引。读取 PDB 的 `ATOM` 记录，提取 $C_\alpha$ 坐标和残基名，用独热编码表示残基类型，并在距离不超过默认阈值 7 Å 的残基之间加入双向边。PDB 字段的格式参见 [PDB 文件简介](https://www.cgl.ucsf.edu/chimera/docs/UsersGuide/tutorials/pdbintro.html)。

In [5]:
# Generate a mapping from amino acids to numbers for one-hot encoding
aa_to_num = dict(
    (aa, i)
    for i, aa in enumerate(
        [
            "ALA",
            "ARG",
            "ASN",
            "ASP",
            "CYS",
            "GLU",
            "GLN",
            "GLY",
            "HIS",
            "ILE",
            "LEU",
            "LYS",
            "MET",
            "PHE",
            "PRO",
            "SER",
            "THR",
            "TRP",
            "TYR",
            "VAL",
            "UNK",
        ]
    )
)


def aa_to_onehot(aa):
    """
    Compute the one-hot vector for an amino acid representing node.

    Arguments
    ---------
    aa: str
        The three-letter code of the amino acid to be represented

    Returns
    -------
    torch.Tensor
        A one-hot tensor encoding the atoms features.
    """
    one_hot = torch.zeros(len(aa_to_num), dtype=torch.float)
    one_hot[aa_to_num[aa]] = 1.0
    return one_hot


def pdb_to_graph(pdb_file_path, max_dist=7.0):
    """
    Extract a graph representation of a protein from the PDB file.

    Arguments
    ---------
    pdb_file_path: str
        Filepath of the PDB file containing structural information on the protein
    max_dist: float
        Distance threshold to apply when computing edges between amino acids

    Returns
    -------
    Tuple[torch.Tensor, torch.Tensor]
        A pair of node features and edges in the PyTorch Geometric format
    """
    # read in the PDB file by looking for the Calpha atoms and extract their amino acid and coordinates based on the positioning in the PDB file
    residues = []
    with open(pdb_file_path, "r") as protein:
        for line in protein:
            if line.startswith("ATOM") and line[12:16].strip() == "CA":
                residues.append(
                    (
                        line[17:20].strip(),
                        float(line[30:38].strip()),
                        float(line[38:46].strip()),
                        float(line[46:54].strip()),
                    )
                )
    # Finally compute the node features based on the amino acids in the protein
    node_feat = torch.stack([aa_to_onehot(res[0]) for res in residues])

    # compute the edges of the protein by iterating over all pairs of amino acids and computing their distance
    edges = []
    for i in range(len(residues)):
        res = residues[i]
        for j in range(i + 1, len(residues)):
            tmp = residues[j]
            if math.dist(res[1:4], tmp[1:4]) <= max_dist:
                edges.append((i, j))
                edges.append((j, i))

    # store the edges in the PyTorch Geometric format
    edges = torch.tensor(edges, dtype=torch.long).T

    return node_feat, edges

### 数据组织

每个样本同时包含蛋白质图、配体图和一个相互作用标签。与只输入一组特征的 MLP 相比，需要让批处理逻辑正确识别两套图的边索引。

#### 单个样本

[PyTorch Geometric](https://pytorch-geometric.readthedocs.io/en/latest/) 内置的 [`Data` 类](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.data.Data.html#torch_geometric.data.Data)通常用来描述一张图。本课扩展这个类，让批处理时的节点数、边数与索引偏移都能分别处理蛋白质图和配体图。

In [6]:
class DTIDataPair(Data):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    @property
    def num_nodes(self):
        return self["lig_x"].size(0) + self["prot_x"].size(0)

    @property
    def num_edges(self):
        return self["lig_edge_index"].size(1) + self["prot_edge_index"].size(1)

    def __inc__(self, key, value, *args, **kwargs):
        """
        Method that is necessary to overwrite for successful batching of DTIDataPair object.
        In case of interest, one can look at this explanation:
        https://pytorch-geometric.readthedocs.io/en/latest/notes/batching.html

        When multiple samples are sent through a network at once, they are aggregated into batches.
        In PyTorch Geometric this is done by copying all n graphs for one batch into one graph with
        n connected components. Because of this, the node ids in the edge_index objects have to be
        changed. As they have to be increased by a fixed offset based on the number of nodes in the
        batch so far, this method computes this offset in case the edge_indices of either the
        proteins or ligands.

        Arguments
        ---------
        key: str
            String name of the field of this class to increment while batching

        Returns
        -------
        torch.Tensor
            A one-element tensor describing how to modify the values when batching.
        """
        if not key.endswith("edge_index"):
            return super().__inc__(key, value, *args, **kwargs)
        lenedg = len("edge_index")
        prefix = key[:-lenedg]
        return self[prefix + "x"].size(0)

#### 数据集

读取相互作用表、配体 SMILES 与蛋白质结构，建立分子图和残基图。根据表中的配对与标签组织样本，完成随机划分后缓存处理结果，以便之后复用。

In [7]:
class DTIDataset(InMemoryDataset):
    def __init__(self, folder_name, file_index):
        self.folder_name = folder_name
        super().__init__(root=folder_name)
        self.data, self.slices = torch.load(self.processed_paths[file_index], weights_only=False)

    @property
    def processed_file_names(self):
        """
        Just store the names of the files where the training split, validation split, and test split are stored.

        Returns
        -------
        List[str]
            A list of filenames where the preprocessed data is stored to not recompute the preprocessing every time.
        """
        return ["train.pt", "val.pt", "test.pt"]

    def process(self):
        """
        This function is called internally in the preprocessing routine of PyTorch Geometric and defined how the dataset of PDB files, ligands, and an interaction table is converted into a dataset of graphs, ready for deep learning.
        """
        # compute all ligand graphs and store them as a dictionary with their names as key and the graphs as values
        ligand_graphs = dict()
        with open(Path(self.folder_name) / "tables" / "ligands.tsv", "r") as data:
            for line in data.readlines()[1:]:
                chembl_id, smiles = line.strip().split("\t")[:2]
                ligand_graphs[chembl_id] = smiles_to_graph(smiles)

        # compute all protein graphs and store them as a dictionary with their names as key and the graphs as values
        protein_graphs = dict(
            [
                (filename[:-4], pdb_to_graph(Path(self.folder_name) / "proteins" / filename))
                for filename in os.listdir(Path(self.folder_name) / "proteins")
            ]
        )

        with open(Path(self.folder_name) / "tables" / "inter.tsv") as inter:
            data_list = []
            for line in inter.readlines()[1:]:
                # read a line with one interaction sample. Extract ligand and protein ID and get their graphs from the dictionaries above
                protein, ligand, y = line.strip().split("\t")
                lig_node_feat, lig_edge_index = ligand_graphs[ligand]
                prot_node_feat, prot_edge_index = protein_graphs[protein]

                # if either ligand or protein are invalid graphs, skip this sample ...
                if lig_node_feat is None or prot_node_feat is None:
                    print(line.strip())
                    continue

                # ... otherwise, create a datapoint using the class from above
                data_list.append(
                    DTIDataPair(
                        lig_x=lig_node_feat,
                        lig_edge_index=lig_edge_index,
                        prot_x=prot_node_feat,
                        prot_edge_index=prot_edge_index,
                        y=torch.tensor(float(y), dtype=torch.float),
                    )
                )

            # shuffle the data, and compute how many samples go into which split
            random.shuffle(data_list)
            train_frac = int(len(data_list) * 0.7)
            test_frac = int(len(data_list) * 0.1)

            # then split the data and store them for later reuse without running the preprocessing pipeline
            train_data, train_slices = self.collate(data_list[:train_frac])
            torch.save((train_data, train_slices), self.processed_paths[0])
            val_data, val_slices = self.collate(data_list[train_frac:-test_frac])
            torch.save((val_data, val_slices), self.processed_paths[1])
            test_data, test_slices = self.collate(data_list[-test_frac:])
            torch.save((test_data, test_slices), self.processed_paths[2])

#### 数据模块

管理训练、验证和测试子集，并提供相应的数据加载器。将多个样本组成一个批次时，需要分别追踪蛋白质节点和配体节点属于哪个样本。

In [8]:
class DTIDataModule:
    def __init__(self, folder_name):
        self.train = DTIDataset(folder_name, 0)
        self.val = DTIDataset(folder_name, 1)
        self.test = DTIDataset(folder_name, 2)

    def train_dataloader(self):
        """
        Create and return a dataloader for the training dataset.

        Returns
        -------
        torch_geometric.loaders.DataLoader
            Dataloader on the training dataset
        """
        return DataLoader(
            self.train, batch_size=64, shuffle=True, follow_batch=["prot_x", "lig_x"]
        )

    def val_dataloader(self):
        """
        Create and return a dataloader for the validation dataset.

        Returns
        -------
        torch_geometric.loaders.DataLoader
            Dataloader on the validation dataset
        """
        return DataLoader(self.val, batch_size=64, shuffle=True, follow_batch=["prot_x", "lig_x"])

    def test_dataloader(self):
        """
        Create and return a dataloader for the test dataset.

        Returns
        -------
        torch_geometric.loaders.DataLoader
            Dataloader on the test dataset
        """
        return DataLoader(self.test, batch_size=64, shuffle=True, follow_batch=["prot_x", "lig_x"])

### 网络

开始实现前面介绍的双编码器和分类模型。

#### GNN 编码器

堆叠 `GINConv` 层，对最后一层节点特征做全局均值池化，再对输出向量进行归一化。蛋白质与配体使用独立的编码器实例。

In [9]:
class GINEncoding(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim=128, output_dim=64, num_layers=3):
        """
        Encoding to embed structural data using a stack of GINConv layers.

        Arguments
        ---------
        input_dim: int
            Size of the feature vector of the data
        hidden_dim: int
            Number of hidden neurons to use when computing the embeddings
        output_dim: int
            Size of the output vector of the final graph embedding after a final mean pooling
        num_layers: int
            Number of layers to use when computing embedding. This includes input and output layers, so values below 3 are meaningless.
        """
        super().__init__()
        self.layers = (
            [
                # define the input layer
                GINConv(
                    nn.Sequential(
                        nn.Linear(input_dim, hidden_dim),
                        nn.PReLU(),
                        nn.Linear(hidden_dim, hidden_dim),
                        nn.BatchNorm1d(hidden_dim),
                    )
                )
            ]
            + [
                # define a number of hidden layers
                GINConv(
                    nn.Sequential(
                        nn.Linear(hidden_dim, hidden_dim),
                        nn.PReLU(),
                        nn.Linear(hidden_dim, hidden_dim),
                        nn.BatchNorm1d(hidden_dim),
                    )
                )
                for _ in range(num_layers - 2)
            ]
            + [
                # define the output layer
                GINConv(
                    nn.Sequential(
                        nn.Linear(hidden_dim, hidden_dim),
                        nn.PReLU(),
                        nn.Linear(hidden_dim, output_dim),
                        nn.BatchNorm1d(output_dim),
                    )
                )
            ]
        )

    def to(self, device):
        """
        Move the network and all its layers to the specified device.

        Arguments
        ---------
        device: torch.device
            The device to move the network to
        """
        for layer in self.layers:
            layer.to(device)
        return self

    def forward(self, x, edge_index, batch):
        """
        Forward a batch of samples through this network to compute the forward pass.

        Arguments
        ---------
        x: torch.Tensor
            feature matrices of the graphs forwarded through the network
        edge_index: torch.Tensor
            edge indices of the graphs forwarded through the network
        batch: torch.Tensor
            Some internally used information, not relevant for the topic of this talktorial
        """
        for layer in self.layers:
            x = layer(x=x, edge_index=edge_index)
        pool = global_mean_pool(x, batch)
        return F.normalize(pool, dim=1)

#### 完整模型

分别计算蛋白质和配体向量，沿特征维度拼接，再输入 MLP。最后一层使用 Sigmoid 输出二分类分数。

In [10]:
class DTINetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()

        # create encoders for both, proteins and ligands
        self.prot_encoder = GINEncoding(21)
        self.lig_encoder = GINEncoding(10)

        # define a simple FNN to compute the final prediction (to bind or not to bind)
        self.combine = torch.nn.Sequential(
            torch.nn.Linear(128, 256),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.2),
            torch.nn.Linear(256, 64),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.2),
            torch.nn.Linear(64, 1),
            torch.nn.Sigmoid(),
        )

    def to(self, device):
        """
        Move the network and all its components to the specified device.

        Arguments
        ---------
        device: torch.device
            The device to move the network to
        """
        self.prot_encoder.to(device)
        self.lig_encoder.to(device)
        self.combine.to(device)
        return self

    def forward(self, data):
        """
        Define the standard forward process of this network.

        Arguments
        ---------
        data: DTIDataPairBatch
            A batch of DTIDataPair samples to be predicted to train on them

        Returns
        -------
        Prediction values for all pairs in the input batch
        """
        # compute the protein embeddings using the protein embedder on the protein data of the batch
        prot_embed = self.prot_encoder(
            x=data.prot_x,
            edge_index=data.prot_edge_index,
            batch=data.prot_x_batch,
        )

        # compute the ligand embeddings using the ligand embedder on the ligand data of the batch
        lig_embed = self.lig_encoder(
            x=data.lig_x,
            edge_index=data.lig_edge_index,
            batch=data.lig_x_batch,
        )

        # concatenate both embeddings and return the output of the FNN
        combined = torch.cat((prot_embed, lig_embed), dim=1)
        return self.combine(combined)

### 训练流程

用 [Adam 优化器](https://pytorch.org/docs/stable/generated/torch.optim.Adam.html#torch.optim.Adam)更新参数，用 BCE 衡量预测误差。官方示例只运行 1 个训练轮次，示例约 10 分钟，实际耗时取决于设备、预处理和数据量。可以增加轮次，观察验证指标是否继续改善。

**阅读实现时可以检查两个细节。** 示例的 `self.layers` 是 Python 列表。若要让编码器层由 PyTorch 自动注册到优化器和检查点中，应使用 `nn.ModuleList` 等模块容器。测试代码的误差累积位于批次循环之后，只计入最后一批。若要比较模型，应使用完整测试集上的汇总指标。

In [11]:
def train(num_epochs=1):
    """
    Implementation of the actual training routine.

    Arguments
    ---------
    num_epochs: int
        Number of epochs to train the model
    """
    # load the data, model, and define the loss function
    dataset = DTIDataModule(DATA / "resources")
    model = DTINetwork().to(device)
    loss_fn = torch.nn.BCELoss()
    optimizer = Adam(model.parameters(), lr=0.0001)
    epoch_train_acc, epoch_train_loss, epoch_val_acc, epoch_val_loss = [], [], [], []

    # train for num_epochs
    for e in range(num_epochs):
        print(f"Epoch {e + 1}/{num_epochs}")

        # set model to training mode
        model.train()

        # perform the actual training
        train_loader = dataset.train_dataloader()
        for b, data in enumerate(train_loader):
            data = data.to(device)
            # compute the models predictions and the loss
            pred = model.forward(data).squeeze()
            loss = loss_fn(pred, data.y.squeeze())

            # perform one step of backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # report some statistics on the training batch
            pred = pred > 0.5
            epoch_train_acc.append(sum(pred.detach().cpu() == data.y.cpu()) / len(data.y))
            epoch_train_loss.append(loss.item())
            print(
                f"\rTraining step {(b + 1)}/{len(train_loader)}: Loss: {epoch_train_loss[-1]:.5f}\tAcc: {epoch_train_acc[-1]:.5f}",
                end="",
            )

        torch.save(model.state_dict(), DATA / f"model_{e + 1}.pth")

        # perform validation of the last training epoch
        val_loader = dataset.val_dataloader()

        # set model to evaluation mode and disable gradient computation
        model.eval()
        with torch.no_grad():
            for b, data in enumerate(val_loader):
                # compute the models predictions and the loss
                data = data.to(device)
                pred = model.forward(data).squeeze()
                loss = loss_fn(pred, data.y.squeeze())

                # report some statistics on the validation batch
                pred = pred > 0.5
                epoch_val_acc.append(sum(pred.detach().cpu() == data.y.cpu()) / len(data.y))
                epoch_val_loss.append(loss.item())
                print(
                    f"\rValidation step {(b + 1)}/{len(val_loader)}: Loss: {epoch_val_loss[-1]:.5f}\tAcc: {epoch_val_acc[-1]:.5f}",
                    end="",
                )

    # test the final model
    print()
    test_loss, test_acc = [], []

    # set model to evaluation mode and disable gradient computation for testing
    model.eval()
    with torch.no_grad():
        for b, data in enumerate(dataset.test_dataloader()):
            # compute the models predictions and the loss
            data = data.to(device)
            pred = model.forward(data).squeeze()
            loss = loss_fn(pred, data.y.squeeze())

        # report some statistics on the validation batch
        pred = pred > 0.5
        test_acc.append(sum(pred.detach().cpu() == data.y.cpu()) / len(data.y))
        test_loss.append(loss.item())
        print(f"\rTesting Loss: {test_loss[-1]:.5f}\tAcc: {test_acc[-1]:.5f}", end="")
    print(
        f"\rTesting: Loss: {(sum(test_loss) / len(test_loss)):.5f}\tAcc: {(sum(test_acc) / len(test_acc)):.5f}"
    )

    fig, ax = plt.subplots(1, 2, figsize=(8, 4), sharey=True)

    ax[0].plot(epoch_train_loss, c="b", label="Train")
    ax[0].plot(
        len(epoch_train_loss) - 1, sum(epoch_val_loss) / len(epoch_val_loss), "rx", label="Val"
    )
    ax[0].set_ylim([0, 1])
    ax[0].set(xlabel="Batches")
    ax[0].set_title("Loss")
    ax[0].grid()

    ax[1].plot(epoch_train_acc, c="b", label="Train")
    ax[1].plot(
        len(epoch_train_acc) - 1, sum(epoch_val_acc) / len(epoch_val_acc), "rx", label="Val"
    )
    ax[1].set_ylim([0, 1])
    ax[1].set(xlabel="Batches")
    ax[1].set_title("Accuracy")
    ax[1].grid()

    plt.legend()
    plt.savefig(IMGS / "train_perf.png")
    plt.clf()


train(1)

/home/rjo21/miniconda3/envs/trash/lib/python3.13/site-packages/torch_geometric/inspector.py:433: DeprecationWarning: Failing to pass a value to the 'type_params' parameter of 'typing._eval_type' is deprecated, as it leads to incorrect behaviour when calling typing._eval_type on a stringified annotation that references a PEP 695 type parameter. It will be disallowed in Python 3.15.
  return typing._eval_type(value, _globals, None)  # type: ignore
/home/rjo21/miniconda3/envs/trash/lib/python3.13/site-packages/torch_geometric/inspector.py:433: DeprecationWarning: Failing to pass a value to the 'type_params' parameter of 'typing._eval_type' is deprecated, as it leads to incorrect behaviour when calling typing._eval_type on a stringified annotation that references a PEP 695 type parameter. It will be disallowed in Python 3.15.
  return typing._eval_type(value, _globals, None)  # type: ignore
/home/rjo21/miniconda3/envs/trash/lib/python3.13/site-packages/torch_geometric/inspector.py:433: Dep

Epoch 1/1
Validation step 64/64: Loss: 0.58500	Acc: 0.71875
Testing: Loss: 0.60836	Acc: 0.69841


<Figure size 800x400 with 0 Axes>

In [12]:
!ls data/resources/processed

/home/rjo21/miniconda3/envs/trash/lib/python3.13/pty.py:95: DeprecationWarning: This process (pid=2404662) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


pre_filter.pt  pre_transform.pt  test.pt  train.pt  val.pt


![首轮训练的分类损失和准确率](./images/train_perf.png)

**图 4.** 首轮训练的损失和准确率。图为随课提供的示例，重新运行的表现可能受拆分、设备和随机过程影响。

## 讨论

图 4 的示例中，损失略降，准确率变化有限。将蛋白质图和配体图各自聚合，再拼接预测，是一种便于理解的概念验证。残基与原子之间的接触位置、局部结合几何、相互作用类型等信息可能影响实际预测。可以结合本课参考资料继续了解更细致的模型、划分与评估策略。

## 自测

1. 为什么用结构和分子图来建模？三维蛋白质结构和原始氨基酸序列分别能提供什么信息？
2. 怎样把蛋白质转换为残基图？节点和边分别由什么定义？
3. 进阶题。为什么需要扩展 PyTorch Geometric 的数据类来表示配对样本？将两条配对记录合成一个批次时，哪些索引需要重新偏移？